# Derivative Pricing Simple Black-Scholes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from DerivModel import train_model, FeedForwardNetwork
from DerivMetrics import evaluate_metrics 
from DerivPlots import scatter_plot, plot_errors
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from tqdm.notebook import tqdm, trange

## Load and Prepare the dataframe

In [ ]:
in_file = 'SPXOptions.csv'
dfopt = pd.read_csv(in_file)

In [ ]:
dfopt

## Train a Model on standard Black-Scholes Inputs and perform hyperparameter tuning

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
#Separate into X and y
X = dfopt[['type', 'strike', 'implied_volatility', 'maturity', 'rate', 'underlying']]
y = dfopt['BlackScholes']

In [ ]:
X['type'] = X['type'].replace({'call': 1, 'put': 0})

In [ ]:
# Split data into train and a temporary set (tmp) first
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.2) 

# Now split the temporary set into test and validation
X_test, X_val, y_test, y_val = train_test_split(X_tmp, y_tmp, test_size=0.5)

In [ ]:
scaler = StandardScaler()

In [ ]:
X_train = scaler.fit_transform(X_train)

In [ ]:
X_test = scaler.transform(X_test)

In [ ]:
X_val = scaler.transform(X_val)

In [ ]:
torch_tensor = torch.from_numpy(X_train).float().to(device)
y_tensor = torch.from_numpy(y_train.to_numpy().copy()).float().to(device)
dataset = TensorDataset(torch_tensor, y_tensor)
batch_size = 512
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

In [ ]:
test_torch_tensor = torch.from_numpy(X_test).float().to(device)
test_y_tensor = torch.from_numpy(y_test.to_numpy().copy()).float().to(device)
test_dataset = TensorDataset(test_torch_tensor, test_y_tensor)
test_data_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

In [ ]:
val_torch_tensor = torch.from_numpy(X_val).float().to(device)
val_y_tensor = torch.from_numpy(y_val.to_numpy().copy()).float().to(device)
val_dataset = TensorDataset(val_torch_tensor, val_y_tensor)
val_data_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

In [ ]:
# Hyperparameters
epochs = 10
lr = 0.001

In [ ]:
# Grid search parameters
hidden_layers_range = [1, 3, 5, 7, 9]
neurons_per_layer_range = [64, 256, 1024, 4096]

In [ ]:
# Placeholder for best model and its performance
best_model = None
best_mse = float('inf')

for num_hidden_layers in hidden_layers_range:
    for neurons_per_layer in neurons_per_layer_range:
        
        # Initialize model, optimizer and loss function
        model = FeedForwardNetwork(input_size=6, 
                                   num_hidden_layers=num_hidden_layers, 
                                   neurons_per_layer=neurons_per_layer).to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        loss_fn = torch.nn.MSELoss()
        
        # Train model
        tqdm_epoch = trange(epochs)
        for epoch in tqdm_epoch:
            model.train()
            
            for batch_X, batch_y in data_loader:
                # Forward pass
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                
                # Backward pass and optimization
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            
        # Evaluate model
        model.eval()
        mse_list = []
        with torch.no_grad():
            for batch_X, batch_y in val_data_loader:
                val_outputs = model(batch_X)
                mse = F.mse_loss(val_outputs.squeeze(), batch_y).item()
                mse_list.append(mse)
                
        avg_mse = sum(mse_list) / len(mse_list)
        print(f"Hidden Layers: {num_hidden_layers}, Neurons/Layer: {neurons_per_layer}, Val Avg. MSE: {avg_mse}")
        
        if avg_mse < best_mse:
            best_mse = avg_mse
            best_model = model

In [ ]:
print("Best Model Configuration:")
print(best_model)

## Build and Train final model

In [ ]:
epochs = 50
no_layers = 5
no_neurons = 4096

In [ ]:
model = FeedForwardNetwork(input_size=6, 
                           num_hidden_layers=no_layers, 
                           neurons_per_layer=no_neurons).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
loss_fn = torch.nn.MSELoss()

In [ ]:
train_errors, test_errors = train_model(model, data_loader, test_data_loader, 
                                        loss_fn, optimizer, epochs)

In [ ]:
plot_file = 'bs_hist.png'
plot_errors(train_errors, test_errors, plot_file)

In [ ]:
metrics, all_results, all_targets = evaluate_metrics(test_data_loader, model)

In [ ]:
metrics

In [ ]:
metric_file = 'bsmetrics.json'
with open(metric_file, 'w') as json_file:
    json.dump(metrics, json_file)

In [ ]:
scatter_filename = 'bsscatter.png'
scatter_plot(all_results, all_targets, scatter_filename)